## 1. Import Python Libraries

In [2]:
# import crim_intervals and specific modules
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
import crim_intervals.corpus_tools as corpus_tools
import music_utils
from music_utils import *
# import other libraries used in the notebooks
import pandas as pd
from IPython.display import display
from pyvis.network import Network
import altair as alt
import plotly_express as px
import os
import re
import seaborn as sns
from collections import Counter

# for use in Jupyter notebook, create a local folder for music files
MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)

else:
    print(MUSDIR, "folder already exists.")

# code for Quarto and the charting tools
alt.renderers.enable('default')
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

import warnings
warnings.filterwarnings('ignore')


    

saved_csv folder already exists.
Music_Files folder already exists.


In [15]:
# function for ngram heatmap:  do not edit!

def ngram_heatmap(piece, combine_unisons_choice, kind_choice, directed, compound, length_choice, include_count, entries_only):
    # find entries for model
    nr = piece.notes(combineUnisons = combine_unisons_choice)
    mel = piece.melodic(df = nr, 
                        kind = kind_choice,
                        directed = directed,
                        compound = compound,
                        end = False)
    
    # this is for entries only
    if entries_only == True:    
        # pass the following ngrams to the plot below as first df
        entry_ngrams = piece.entries(df = mel, 
                                    n = length_choice, 
                                    thematic = True, 
                                    anywhere = True,
                                    exclude = ['Rest'])
        # pass the ngram durations below to the plot as second df
        entry_ngrams_duration = piece.durations(df = mel, 
                                            n =length_choice, 
                                            mask_df = entry_ngrams)
        # make the heatmap
        chart = viz.plot_ngrams_heatmap(entry_ngrams, 
                                         entry_ngrams_duration, 
                                         selected_patterns=[], 
                                         voices=[],  
                                         includeCount=include_count)
        
        return chart
    
    # this is for ALL mel ngrams (if entries is False in form)
    else:
        mel_ngrams = piece.ngrams(df = mel, n = length_choice, exclude = ['Rest'])
        mel_ngrams_duration = piece.durations(df = mel, 
                                          n =length_choice)
        
        chart = viz.plot_ngrams_heatmap(mel_ngrams, 
                                         mel_ngrams_duration, 
                                         selected_patterns=[], 
                                         voices=[], 
                                         includeCount=include_count)
        
        # # mel_ngrams_detail = piece.detailIndex(mel_ngrams, offset = False)  
        return chart

#define the function to convert tuples to strings
def convert_tuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out  
# Create a color map dictionary that maps each cadence type to a specific color
cadence_color_dict = {
    'Authentic': '#1f77b4',  # blue
    'Clausula Vera': '#ff7f0e',  # orange
    'Evaded Authentic': '#2ca02c',  # green
    'Leaping Contratenor': '#d62728',  # red
    'Phrygian Clausula Vera': '#9467bd',  # purple
    'Phrygian': '#8c564b',  # brown
    'Evaded Clausula Vera': '#e377c2',  # pink
    'Double Leading Tone': '#7f7f7f',  # gray
    'Abandoned Clausula Vera': '#bcbd22',  # olive
    'Quince': '#17becf',  # cyan
    'Altizans Only': '#aec7e8',  # light blue
    'Abandoned Authentic': '#ffbb78'  # light orange
}

# List of values for CVFs
values = ['Cx', 'CB', 'CTB', 'TCB', 'zCB', 'CT', 'CTA', 'CTx', 'TC', 'TCA',
          'TCx', 'ACT', 'CAT', 'CTb', 'TCb', 'CTL', 'CzB', 'CxT', 'zCT',
          'Ct', 'tCx']

# Get the default color palette
color_palette = px.colors.qualitative.Plotly

# Create a color dictionary
cvf_color_dict = {value: color for value, color in zip(values, color_palette)}


# tone order for charts

custom_tone_order = ['E-', 'B-', 'F', 'C', 'G', 'D', 'A', 'E', 'B']  # Your desired o


def radar_note_plot(weighted_notes_df, pitch_class_order='Fifths',
                    exclude_rests=True, limit_to_active=True, color_grouping='title'):
    if exclude_rests:
        weighted_notes_df = weighted_notes_df[weighted_notes_df['pitch_class'] != 'Rest']
    if limit_to_active:
        weighted_notes_df = weighted_notes_df[weighted_notes_df['scaled'] > 0]

    pc_order = NOTE_PC_ORDERS[pitch_class_order]
    active_pcs = [pc for pc in pc_order if pc in weighted_notes_df['pitch_class'].values]

    weighted_notes_df = weighted_notes_df.copy()
    weighted_notes_df['pitch_class'] = pd.Categorical(
        weighted_notes_df['pitch_class'], categories=active_pcs, ordered=True
    )
    weighted_notes_df = weighted_notes_df.sort_values('pitch_class')

    fig = px.line_polar(
        weighted_notes_df,
        r='scaled',
        theta='pitch_class',
        color=color_grouping,
        line_close=True,
        range_r=[0, weighted_notes_df['scaled'].max() * 1.1],
        markers=True,
        category_orders={
            'pitch_class': active_pcs,
            color_grouping: list(weighted_notes_df[color_grouping].unique())
        },
        color_discrete_sequence=contrasting_colors[:len(weighted_notes_df[color_grouping].unique())]
    )
    fig.update_traces(fill='toself', mode='markers+lines', opacity=.7)
    fig.update_layout(
        showlegend=True,
        legend=dict(title=color_grouping, orientation='h', yanchor='bottom', y=-0.5, xanchor='right', x=1),
        height=600, width=600,
        title=f'Weighted Note Distribution in Corpus ({pitch_class_order} order)'
    )
    fig.show()
    fig.write_html("charts/radar_note_plot.html")


def cadence_radar(cadences, tone_ordering='Thirds', limit_to_active=False):
    FIFTHS    = {'C':0,'G':1,'D':2,'A':3,'E':4,'B':5,'F#':6,'C#':7,'A-':8,'E-':9,'B-':10,'F':11}
    THIRDS    = {'C':0,'E':1,'G':2,'B-':3,'D':4,'F':5,'A':6,'C#':7}
    CHROMATIC = {'C':0,'C#':1,'D-':1,'D':2,'E-':3,'E':4,'F':5,'F#':6,'G-':6,'G':7,'A-':8,'A':9,'B-':10,'B':11}
    CATEGORY_ORDER = FIFTHS if tone_ordering == 'Fifths' else THIRDS if tone_ordering == 'Thirds' else CHROMATIC

    grouped = cadences.groupby(['Title', 'Tone']).size().reset_index(name='count')
    title_sums = grouped.groupby('Title')['count'].sum()
    grouped['Percentage'] = (grouped['count'] / grouped['Title'].map(title_sums)) * 100

    tone_order  = sorted(CATEGORY_ORDER.keys(), key=lambda x: CATEGORY_ORDER[x])
    active_tones = [t for t in tone_order if t in grouped['Tone'].values] if limit_to_active else tone_order

    full_index = pd.MultiIndex.from_product([grouped['Title'].unique(), active_tones], names=['Title', 'Tone'])
    grouped = (
        grouped[grouped['count'] > 0]
        .set_index(['Title', 'Tone'])[['count', 'Percentage']]
        .reindex(full_index, fill_value=0)
        .reset_index()
    )
    if limit_to_active:
        grouped = grouped[grouped['count'] > 0]

    tone_rank = {t: i for i, t in enumerate(active_tones)}
    grouped = grouped.assign(_sort=grouped['Tone'].map(tone_rank)).sort_values(['Title', '_sort']).drop(columns='_sort').reset_index(drop=True)
    grouped['Tone'] = pd.Categorical(grouped['Tone'], categories=active_tones, ordered=True)

    fig = px.line_polar(grouped, r='Percentage', theta='Tone', line_close=True,
                        color='Title', markers=True, category_orders={'Tone': active_tones})
    fig.update_traces(fill='toself', line=dict(width=2))
    fig.update_layout(
        width=800, height=600,
        legend=dict(orientation='h', yanchor='bottom', y=-0.4, xanchor='center', x=0.5,
                    title=dict(text='Titles', side='top', font_size=12),
                    itemsizing='constant', itemwidth=30,
                    bordercolor='black', borderwidth=1, bgcolor='rgba(255,255,255,0.8)'),
        title=dict(text=f'Relative Distribution of Cadence Tones in Corpus ordered by {tone_ordering}', x=0.5),
        polar=dict(radialaxis=dict(visible=True, title='Percentage')),
        margin=dict(b=120)
    )
    fig.show()
    fig.write_html("charts/radar_cadence_plot.html")


# =============================================================================
# VISUALIZATION: SPECIES + CADENCES
# =============================================================================

def plot_species_cadences(species_df, cadences_df, composer, title):
    """
    Plot species intervals and cadences by progress in a two-panel chart.

    Args:
        species_df:   output of find_species()
        cadences_df:  piece.cadences() with 'Composer' and 'Title' columns added
        composer:     piece composer string (for chart title)
        title:        piece title string (for chart title)
    """
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=('Species: Octave by Progress', 'Cadences: Tone by Progress')
    )

    # Normalize Progress to 0–1 for both datasets so they share the same x-axis scale
    species_df  = species_df.copy()
    cadences_df = cadences_df.copy()
    for df, col in [(species_df, 'Progress'), (cadences_df, 'Progress')]:
        lo, hi = df[col].min(), df[col].max()
        if hi > lo:
            df[col] = (df[col] - lo) / (hi - lo)

    octave_order = ['EbBbEb', 'BbFBb', 'FCF', 'CGC', 'GDG', 'DAD', 'AEA', 'EBE', 'BF#B']
    fig_sp = px.scatter(
        species_df, x='Progress', y='Octave', color='Voice',
        hover_data=['composer', 'title', 'Measure', 'Interval', 'Species'],
        color_discrete_sequence=contrasting_colors[:len(species_df['Voice'].unique())],
        category_orders={'Octave': octave_order}
    )
    for trace in fig_sp.data:
        trace.showlegend = True
        fig.add_trace(trace, row=1, col=1)
    fig.update_yaxes(title_text='Octave', categoryorder='array', categoryarray=octave_order, row=1, col=1)

    tone_order = ['B-', 'F', 'C', 'G', 'D', 'A', 'E']
    fig_cad = px.scatter(
        cadences_df, x='Progress', y='Tone', color='CadType',
        hover_data=['Composer', 'Title', 'CadType', 'Tone'],
        color_discrete_sequence=contrasting_colors[len(species_df['Voice'].unique()):],
        category_orders={'Tone': tone_order}
    )
    for trace in fig_cad.data:
        trace.marker.symbol = 'diamond'
        trace.marker.size = 10
        trace.marker.opacity = 0.8
        trace.showlegend = True
        fig.add_trace(trace, row=2, col=1)

    fig.update_traces(marker=dict(size=8, opacity=0.7), row=1, col=1)
    fig.update_yaxes(title_text='Tone', categoryorder='array', categoryarray=tone_order, row=2, col=1)
    fig.update_yaxes(title_text='Octave', tickmode='linear', dtick=1, row=1, col=1)

    x_min = min(species_df['Progress'].min(), cadences_df['Progress'].min())
    x_max = max(species_df['Progress'].max(), cadences_df['Progress'].max())
    fig.update_xaxes(range=[x_min, x_max], autorange=False)
    fig.update_xaxes(title_text='Progress', row=2, col=1)
    fig.update_layout(
        height=600, width=800,
        legend=dict(x=1.02, y=1.0, xanchor='left', yanchor='top'),
        margin=dict(r=200),
        title=f'Species and Cadences by Progress in {composer} : {title}',
        legend_title='Legend'
    )
    fig.show()


# Voice Range Chart


def find_mode_range(df, final_value, voice_value, top_n=7):
    """
    Finds the range of notes in a given voice to help distinguish modal types.

    df:          corpus notes DataFrame (from corpus_note_durs with pitch_class=False)
    final_value: the final tone of each piece
    voice_value: which voice to check
    top_n:       number of highest-percentage notes to use for range calculation
    """
    filtered_df = df[
        (df['Final'] == final_value) &
        (df['Voice'] == voice_value) &
        (df['Notes'] != 'Rest')
    ]

    top_durations = filtered_df.groupby(['Title', 'Voice'], observed=True).apply(
    lambda x: x.nlargest(top_n, 'Percentage'), include_groups=False).reset_index(level=['Title', 'Voice']).reset_index(drop=True)

    # Build note-position mapping
    base_notes = ['C', 'D', 'E', 'F', 'G', 'A', 'B', 'B-']
    octaves = list(range(2, 6))
    note_positions = {}
    position = 0
    for octave in octaves:
        for note in base_notes:
            note_positions[f"{note}{octave}"]  = position
            note_positions[f"{note}-{octave}"] = position - 0.5
            note_positions[f"{note}b{octave}"] = position - 0.5
            note_positions[f"{note}#{octave}"] = position + 0.5
            position += 1

    def get_note_position(note):
        if note in note_positions:
            return note_positions[note]
        if '-' in note:
            parts = note.split('-')
            if len(parts) == 2:
                alt = f"{parts[0]}-{parts[1]}"
                if alt in note_positions:
                    return note_positions[alt]
        if '#' in note:
            parts = note.split('#')
            if len(parts) == 2:
                alt = f"{parts[0]}#{parts[1]}"
                if alt in note_positions:
                    return note_positions[alt]
        print(f"Warning: Could not determine position for note: {note}")
        return -1000

    top_durations['NotePosition'] = top_durations['Notes'].apply(get_note_position)

    result = []
    for (title, voice), group in top_durations.groupby(['Title', 'Voice'], observed=True):
        if len(group) > 0:
            sorted_group = group.sort_values('NotePosition')
            lowest_note  = sorted_group.iloc[0]['Notes']
            highest_note = sorted_group.iloc[-1]['Notes']
            title_rows   = df[df['Title'] == title]
            composer = title_rows['Composer'].iloc[0] if not title_rows.empty else 'Unknown'
            final    = title_rows['Final'].iloc[0]    if not title_rows.empty else 'Unknown'
            result.append({
                'Composer':    composer,
                'Title':       title,
                'Voice':       voice,
                'Final':       final,
                'LowestNote':  lowest_note,
                'HighestNote': highest_note,
                'Range':       f"{lowest_note} to {highest_note}"
            })

    return pd.DataFrame(result)


def voice_range_chart(corpus_pitch_data, recta_order, top_n=7):

    final_value = corpus_pitch_data['Final'].iloc[0]
    designated_voice = '2'

    # Run find_mode_range for every voice and combine
    voices = corpus_pitch_data['Voice'].unique()
    all_voice_ranges = []
    for voice in voices:
        voice_df = find_mode_range(corpus_pitch_data, final_value=final_value, voice_value=voice, top_n=top_n)
        if not voice_df.empty:
            all_voice_ranges.append(voice_df)

    corpus_voice_ranges = pd.concat(all_voice_ranges, ignore_index=True)

    # Add positional columns needed by the chart
    corpus_voice_ranges['Lowest_pos'] = corpus_voice_ranges['LowestNote'].apply(
        lambda n: recta_order.index(n) if n in recta_order else 0
    )
    corpus_voice_ranges['Highest_pos'] = corpus_voice_ranges['HighestNote'].apply(
        lambda n: recta_order.index(n) if n in recta_order else 0
    )
    corpus_voice_ranges['Range_width'] = (
        corpus_voice_ranges['Highest_pos'] - corpus_voice_ranges['Lowest_pos']
    )

    # Force voice to string for chart labels
    corpus_voice_ranges['Voice'] = corpus_voice_ranges['Voice'].astype(str)

    # Movement order: titles without a movement keyword are chanson models → rank 0
    mass_movements = ['Kyrie', 'Gloria', 'Credo', 'Sanctus', 'Agnus Dei']

    def title_rank(title):
        for i, movement in enumerate(mass_movements):
            if movement in title:
                return i + 1
        return 0  # no movement keyword → chanson model, sorts first

    # Find lowest note of designated voice per piece, for sorting within groups
    designated_voice_notes = {}
    for _, row in corpus_voice_ranges.iterrows():
        if row['Voice'] == designated_voice:
            designated_voice_notes[row['Title']] = row['LowestNote']

    sorted_titles = sorted(
        designated_voice_notes.keys(),
        key=lambda x: (
            title_rank(x),                                # primary: model first, then movements
            recta_order.index(designated_voice_notes[x])  # secondary: lowest note
        )
    )

    # Create combined Title+Voice label, voices sorted numerically within each piece
    corpus_voice_ranges['Title_Voice'] = (
        corpus_voice_ranges['Title'] + '  [V' + corpus_voice_ranges['Voice'] + ']'
    )

    sorted_title_voice = []
    for title in sorted_titles:
        piece_rows = corpus_voice_ranges[corpus_voice_ranges['Title'] == title]
        for voice in sorted(piece_rows['Voice'].unique(), key=lambda v: int(v)):
            label = f"{title}  [V{voice}]"
            if label not in sorted_title_voice:
                sorted_title_voice.append(label)

    # Create figure
    fig = px.bar(
        corpus_voice_ranges,
        x='Range_width',
        y='Title_Voice',
        base='Lowest_pos',
        orientation='h',
        color='Voice',
        color_discrete_map={
            '1': 'rgba(255, 99, 71, 0.5)',
            '2': 'rgba(54, 162, 235, 0.5)',
            '3': 'rgba(100, 20, 100, 0.3)'
        },
        hover_data=['Title', 'Voice', 'Range'],
        category_orders={'Title_Voice': sorted_title_voice},
        labels={'Title_Voice': 'Piece / Voice'}
    )

    _ref_by_final = {
        'D': ['D3', 'G3', 'D4', 'G4', 'D5', 'G5'],
        'E': ['E3', 'C3', 'E4', 'C4', 'E5', 'C5'],
        'F': ['C3', 'F3', 'C4', 'F4', 'C5', 'F5'],
        'G': ['D3', 'G3', 'D4', 'G4', 'D5', 'G5'],
        'A': ['E3', 'A3', 'E4', 'A4', 'E5', 'A5'],
        'C': ['C3', 'G3', 'C4', 'G4', 'C5', 'G5'],
    }
    reference_points = _ref_by_final.get(final_value, ['D3', 'G3', 'D4', 'G4', 'D5', 'G5'])
    shapes = []
    annotations = []
    for point in reference_points:
        if point in recta_order:
            x_position = recta_order.index(point)
            shapes.append(dict(
                type='line',
                x0=x_position, y0=0,
                x1=x_position, y1=len(sorted_title_voice),
                line=dict(dash='dot', color='gray', width=1)
            ))
            annotations.append(dict(
                x=x_position,
                y=len(sorted_title_voice) - 1,
                text=point,
                showarrow=False,
                font=dict(size=12)
            ))

    fig.update_layout(
        title=f'Voice Ranges in Pieces with Final {final_value}',
        xaxis=dict(
            title='Note Range',
            tickmode='array',
            tickvals=list(range(len(recta_order))),
            ticktext=recta_order,
            tickangle=45
        ),
        yaxis=dict(
            title='',
            autorange='reversed',
            categoryorder='array',
            categoryarray=sorted_title_voice
        ),
        height=900,
        width=1000,
        bargap=0.3,
        bargroupgap=0.1,
        margin=dict(l=250, r=100, t=100, b=50),
        shapes=shapes,
        annotations=annotations
    )

    fig.show()
    fig.write_html("charts/corpus_voice_ranges.html")

# =============================================================================
# CONSTANTS
# =============================================================================

pitch_order = ['A#1', 'B1',
    'C2', 'C#2', 'D2', 'D#2','E-2', 'E2', 'E#2', 'F2', 'F#2', 'G-2', 'G2', 'G#2', 'A-2', 'A2', 'A#2','B-2', 'B2', 'B#2',
    'C3', 'C#3', 'D-3','D3', 'D#3', 'E-3','E3', 'E#3', 'F3', 'F#3', 'G-3', 'F##3', 'G3', 'G#3', 'A-3', 'A3', 'A#3', 'B-3','B3', 'B#3',
    'C4', 'C#4', 'D-4','D4', 'D#4','E-4', 'E4', 'F-4', 'E#4', 'F4', 'F#4', 'G-4', 'F##4', 'G4', 'G#4', 'A-4','A4', 'A#4', 'B-4', 'B4', 'B#4',
    'C5', 'C#5','C##5', 'D-5','D5', 'D#5', 'E-5','E5', 'F-5','E#5','F5', 'F#5', 'G-5', 'F##5','G5', 'G#5', 'A-5', 'A5', 'A#5', 'B-5', 'B5',
    'C6']

recta_order = ['D2', 'E-2', 'E2', 'F2', 'F#2', 'G2', 'G#2', 'A2', 'B-2', 'B2',
               'C3', 'C#3','D3', 'D#3', 'E-3','E3', 'F3', 'F#3',  'G3', 'G#3', 'A-3', 'A3', 'B-3','B3',
               'C4', 'C#4', 'D-4','D4', 'D#4','E-4', 'E4', 'F4', 'F#4', 'G-4',  'G4', 'G#4', 'A-4','A4',  'B-4', 'B4',
               'C5', 'C#5','D-5','D5', 'D#5', 'E-5','E5','F5', 'F#5', 'G-5', 'G5', 'G#5', 'A-5', 'A5',  'B-5', 'B5',
               'C6']

pitch_class_order_no_rests = ['C', 'C#', 'D-','D', 'D#', 'E-', 'E', 'F-', 'E#', 'F', 'F#', 'G-', 'F##', 'G', 'G#', 'A-','A', 'A#', 'B-', 'B', 'B#']
pitch_class_order_with_rests = pitch_class_order_no_rests + ['Rest']
pitch_class_order = pitch_class_order_with_rests

category_order = {
    'C': 0, 'C#': 1, 'Db': 2, 'D': 3, 'D#': 4, 'Eb': 5, 'E': 6, 'F': 7, 'F#': 8,
    'G': 9, 'G#': 10, 'A': 11, 'A#': 12, 'Bb': 13, 'B': 14, 'Rest': 15
}

REST_TOKENS = {'r', 'rest', 'Rest', '-', ''}

NOTE_PC_CHROMATIC = ['C', 'C#', 'D-', 'D', 'D#', 'E-', 'E', 'F', 'F#', 'G-', 'G', 'G#', 'A-', 'A', 'A#', 'B-', 'B', 'F-', 'E#', 'F##', 'B#', 'Rest']
NOTE_PC_FIFTHS    = ['C', 'G', 'D', 'A', 'E', 'B', 'F#', 'C#', 'A-', 'E-', 'B-', 'F', 'G#', 'D#', 'A#', 'D-', 'G-', 'F-', 'E#', 'B#', 'F##', 'Rest']
NOTE_PC_DIATONIC  = ['C', 'D', 'E', 'F', 'G', 'A', 'B', 'B-', 'E-', 'A-', 'D-', 'C#', 'F#', 'G#', 'D#', 'A#', 'G-', 'F-', 'E#', 'F##', 'B#', 'Rest']
NOTE_PC_ORDERS    = {'Chromatic': NOTE_PC_CHROMATIC, 'Fifths': NOTE_PC_FIFTHS, 'Diatonic': NOTE_PC_DIATONIC}

contrasting_colors = [
    '#636EFA', '#DC267F', '#009E73', '#FFB000', '#977277',
    '#EC4899', '#48BB78', '#ED8936', '#2563EB', '#8338EC',
    '#FF922B', '#06D6A0', '#EF4444', '#F97316', '#84CC16',
    '#3B82F6', '#A855F7', '#22C55E', '#EA580C', '#94A3B8',
]

def standardize_note(note):
    if '-' in note:
        return note.replace('-', 'b')
    return note



# 2. Import a Score

Here we find a piece to explore.  

Load it remotely:

```python
path = 'https://crimproject.org/mei/CRIM_Model_0019.mei'

piece = importScore(path)
```

Load it locally:

```python
path = "MEI/Bach_BWV_0772.mei"
piece = importScore(path)
```




In [16]:
path = 'https://crimproject.org/mei/CRIM_Model_0019.mei'
piece = importScore(path)


piece.metadata

{'title': 'Veni sponsa Christi',
 'composer': 'Giovanni Pierluigi da Palestrina',
 'date': None}

# 3. Count and Chart the Notes

### 3a. Here we count the notes and report the distributions.

In [17]:
nr = piece.notes()
nr = piece.numberParts(nr).melt()

# count the notes? first we 'melt' the data to make it 'long data'
# melted_notes = notes_rests_num_parts.melt()
melted_notes = nr.rename(columns={'variable': 'voice', 'value': 'pitch'})
melted_notes = melted_notes.dropna()
counted_notes = Counter(melted_notes['pitch'])
counted_notes

Counter({'D5': 26,
         'B4': 46,
         'E5': 11,
         'C5': 38,
         'A4': 61,
         'G4': 74,
         'Rest': 38,
         'G5': 2,
         'F5': 7,
         'F4': 45,
         'G#4': 2,
         'F#4': 5,
         'E4': 62,
         'D4': 66,
         'B3': 39,
         'C4': 58,
         'A3': 38,
         'G3': 33,
         'C#4': 1,
         'E3': 11,
         'F3': 16,
         'F#3': 1,
         'C3': 5,
         'D3': 9})

### 3b.  Bar Chart of Notes, Voices and Ranges

In [18]:
# pitch order is imported from sorting_lists.py
pitch_order = pitch_order

# chart of pitches by voice
grouped_data = melted_notes.groupby(['pitch', 'voice']).size().reset_index(name='count')

# Create the stacked bar chart
chart_title = "Distribution of Notes in " + piece.metadata['title']
fig = px.bar(grouped_data,
             x='pitch',
             y='count',
             color='voice',
             barmode='stack',
             title=chart_title,
             # note that we define pitch_order in the helpers file!
             category_orders={'pitch': pitch_order})

fig.update_layout(xaxis_title="Note", 
                  yaxis_title="Count",
                  legend_title='Voice',
                  height=500,
                  width=750,
                  xaxis={'categoryorder': 'array', 'categoryarray': pitch_order})

fig.show()

## 3c.  Weighted Notes Radar Plot

What about the **pitch class distribution**?  That is:  the 'class' of tone, quite apart from the octave in which it appears? 

And why not also consider the **duration of each note**, which will tell us something more useful about how the piece sounds, in terms of pitch space.

And it might be interesting to see these **weighted note** distributions as a radial (or 'radar' plot), giving us a visual impression of the overall tonal footprint of the piece.

Here will use a new CRIM function that is part of the `corpus` library in CRIM:


`corpus.note_weights(include_rests=True, key_sig=False, include_final=False)`





In [19]:
# just ONE piece for testing

test_piece = ['https://crimproject.org/mei/CRIM_Model_0012.mei']

corpus = CorpusBase(test_piece)

# define pitch collection from sorting_lists.py

pitch_sequence = pitch_class_order_with_rests

weighted_notes = corpus.note_weights(include_rests=True, key_sig=False, include_final=False)
weighted_notes = weighted_notes.rename(columns={'index': 'pitch_class'})

# the pitch_class_order_with_rests is defined in the code base
weighted_notes['pitch_class'] = pd.Categorical(weighted_notes['pitch_class'], categories=pitch_sequence, ordered=True)

# Sort the DataFrame
weighted_notes_sorted = weighted_notes.sort_values('pitch_class')
weighted_notes_sorted['pitch_class'] =weighted_notes_sorted['pitch_class'].astype(str)

# Reset the index and correct flat names--the 'standardize note' function is in the code base
weighted_notes_sorted = weighted_notes_sorted.reset_index(drop=True)
weighted_notes_sorted['pitch_class']= weighted_notes_sorted['pitch_class'].map(standardize_note)


# save csv for later use
weighted_notes_sorted.to_csv('full_corpus_note_weights.csv')

# display in descending order of weight
weighted_notes_sorted.sort_values('scaled', ascending=False)

,pitch_class,count,scaled,composer,title
2,D,339.0,0.2057,Pierre Sandrin,Voulant honneur
7,G,286.5,0.1738,Pierre Sandrin,Voulant honneur
5,F,215.0,0.1305,Pierre Sandrin,Voulant honneur
9,Bb,213.0,0.1292,Pierre Sandrin,Voulant honneur
8,A,211.5,0.1283,Pierre Sandrin,Voulant honneur
0,C,181.0,0.1098,Pierre Sandrin,Voulant honneur
10,Rest,96.0,0.0583,Pierre Sandrin,Voulant honneur
4,E,79.0,0.0479,Pierre Sandrin,Voulant honneur
6,F#,16.0,0.0097,Pierre Sandrin,Voulant honneur
1,C#,8.0,0.0049,Pierre Sandrin,Voulant honneur


In [20]:
# set title and size
chart_title = 'Weighted Note Distribution in ' + piece.metadata['title']
height = 600
width = 600

# to close the diagram, we add the first row again at the end:
weighted_notes_sorted.iloc[-1] = weighted_notes_sorted.iloc[0]
weighted_notes_sorted

# pitch class order is imported from sorting_lists.py.  We could also omit 

pitch_sequence = pitch_class_order_with_rests

# Single piece plot
fig = px.line_polar(
    weighted_notes_sorted,
    r='scaled',
    theta='pitch_class',
    category_orders={'pitch': pitch_sequence},
    line_close=True,
    range_r=[0, weighted_notes_sorted['scaled'].max() * 1.1],
    markers=True,
    title = chart_title,
    height=height,
    width=width
    
)

fig.update_traces(
    fill='toself',
    fillcolor='rgba(31, 119, 180, 0.3)',
    line=dict(width=2))

fig.show()  

# Save the figure as an interactive HTML file
fig.write_html("weighted_notes.html")

# 4 Ngrams:  Melodic

### 4a. Here are all the mel ngs.

In [21]:

mel = piece.melodic(kind='d', end=False)
mel = piece.numberParts(df=mel)
mel_ng = piece.ngrams(df=mel, n=3)
# clean up the tuples as strings
mel_ng_clean = mel_ng.map(convert_tuple)
mel_ng_clean


,1,2,3,4
0.0,"-3, 3, 2",,,
8.0,"3, 2, -2","-3, 3, 2",,
12.0,"2, -2, -3",,,
16.0,"-2, -3, 2","3, 2, -2",,
20.0,"-3, 2, 2","2, -2, -3",,
...,...,...,...,...
514.0,"-2, -2, -2",,"3, -2, -2",
516.0,,"-2, -2, 1","-2, -2, -2",
517.0,"-2, -2, 1",,,
518.0,"-2, 1, -2",,,


### 4b. How Many of Each ngram?

In [22]:
mel_ng_melt = mel_ng_clean.melt()
mel_ng_melt_clean = mel_ng_melt[mel_ng_melt['value'] != '']
mel_ng_counts = pd.DataFrame(mel_ng_melt_clean['value'].value_counts()).reset_index()

# the ones that are not unique (and thus repeat)
mel_ng_counts[mel_ng_counts['count'] > 1].head(10)

,value,count
0,"2, 2, 2",63
1,"-2, -2, -2",49
2,"2, 2, -2",42
3,"-2, -2, 2",40
4,"2, -2, -2",31
5,"-3, 2, 2",27
6,"-2, 2, 2",24
7,"3, -2, -2",16
8,"-2, -3, 2",13
9,"2, -2, -3",12


## 4c. Where are These Ngrams?

Can we begin to think about where the **fugas** are based on the **offset differences** between the ngrams?  Are the fugas likely to share the same time intervals between entries?

Based on what you see here, what do you **expect to find in the score**?

In [23]:
# define parameters

kind = 'd'
end=False
length = 3


mel = piece.melodic(kind=kind, end=end)
mel = piece.numberParts(df=mel)
mel_ng = piece.ngrams(df=mel, n=length)
# clean up the tuples as strings
mel_ng_clean = mel_ng.map(lambda x: piece.tuple_to_string(x, separator='_'))



# stack all the ngs
mel_ng_stacked = mel_ng_clean.stack()

# find just one of the ngs

ngram_target = '3_-2_-2'
selected_ng = mel_ng_stacked[mel_ng_stacked == ngram_target]

# use detailIndex to report offset, measure, and beat information
selected_ng_locations = piece.detailIndex(df=selected_ng.unstack().fillna(''), offset=True)
selected_ng_locations

1        2        3        4
Measure Beat Offset                                    
39.0    1.0  304.0            3_-2_-2                  
        3.0  308.0                     3_-2_-2         
43.0    1.0  336.0                              3_-2_-2
45.0    1.0  352.0            3_-2_-2                  
47.0    1.0  368.0   3_-2_-2                           
49.0    3.0  388.0                              3_-2_-2
51.0    2.0  402.0            3_-2_-2                  
52.0    4.0  414.0   3_-2_-2                           
54.0    2.0  426.0                              3_-2_-2
56.0    2.0  442.0                     3_-2_-2         
57.0    4.0  454.0            3_-2_-2                  
60.0    2.0  474.0                              3_-2_-2
62.0    4.0  494.0   3_-2_-2                           
63.0    2.0  498.0            3_-2_-2                  
65.0    1.0  512.0   3_-2_-2                           
        2.0  514.0                     3_-2_-2

## 4d. Ngrams in Time

Here we show the ngrams as the appear in time, from the beginning (at the left) to the end (at the right) of the motet. 

- Each **ngram** is associated with a distinct color, and the relative placement of the ngrams among the voices can be used to imagine the kinds of contrapuntal patterns heard at each moment in the piece.

- The heatmap charts are dynamic:  you can click on a given color to isolate that ngram to see where it appears in the piece.

- Note:  some colors repeat or are hard to distinguish from each other.  But the 'key' at the top and right make it clear which color corresponds to which ngram.

As a reminder, here are the ngram settings used:

```python
entries_only = True
combine_unisons_choice = False
kind_choice ='d'
directed = True
compound = False
length_choice = 4
include_count = False
```

Then to run the maps for any set of files:


```python
for file in file_list:
    piece = importScore(file)

    chart = ngram_heatmap(piece, combine_unisons_choice, kind_choice, directed, compound, length_choice, include_count)
    print(piece.metadata)
    display(chart)
```

In [24]:

# the ngram conditions
entries_only = True
combine_unisons_choice = False
kind_choice ='d'
directed = True
compound = False
length_choice = 3
include_count = True

# calling the function on your file list:

# for filename in my_pieces:
filename = 'https://crimproject.org/mei/CRIM_Model_0019.mei'
piece = importScore(filename)

chart = ngram_heatmap(piece, combine_unisons_choice, kind_choice, directed, compound, length_choice, include_count, entries_only)


print(piece.metadata)
display(chart)
        
    

{'title': 'Veni sponsa Christi', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': None}


alt.VConcatChart(...)

# 5. Presentation Types with Linked Examples

Here we use another tool to identify the Presentation Types formed by the ngrams, according to Peter Schubert's useful taxonomy of Periodic Entries, Imitative Duos, and Fugas.

We also return these with useful metadata and the possibility to show them as engraved verovio scores with highlights.



In [25]:
# here we load

baseUrl = 'https://crimproject.org/mei/'

# now specify YOUR piece:
my_piece = 'CRIM_Model_0019.mei'


# and combine them
piece_url = baseUrl + my_piece

# load the piece from the web
piece = importScore(piece_url)

# find p types
pt = piece.presentationTypes(kind='d',
                             limit_to_entries = True,
                        head_flex=0,
                        body_flex = 0,
                        include_hidden_types = False,
                        combine_unisons = False,
                        melodic_ngram_length = 3)

# reorganize columns
cols = ['Composer', 'Title', 'Presentation_Type', 'Soggetti', 'Voices', 
'Number_Entries', 'Measures_Beats', 'Melodic_Entry_Intervals', 'Time_Entry_Intervals',	
'Flexed_Entries', 'Parallel_Entries','Parallel_Voice', 'Count_Non_Overlaps', 'Offsets', 'Progress']

pt = pt[cols]

# and return the results with a link to the example
piece.linkExamples(df = pt, piece_url=piece_url)


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F1%2C11%2C12%2C13%2C14%2C15%2C16%2C17%2C18%2C19%2C2%2C3%2C4%2C6%2C7%2C8%2C9%2F1%2C2%2C2%2C2%2C1%2C1%2C1%2C4%2C4%2C4%2C1%2B2%2C1%2B2%2C2%2C3%2C3%2B4%2C3%2B4%2C4%2F%40all%2C%403.0-end%2C%40all%2C%401-3.0%2C%40all%2C%40all%2C%401-1.0%2C%40all%2C%40all%2C%401-1.0%2C%40all%2B%40all%2C%401-1.0%2B%40all%2C%401-1.0%2C%40all%2C%40all%2B%40all%2C%401-1.0%2B%40all%2C%401-1.0%2Fhighlight&measure_range=1-19&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus, Altus, Cantus, Bassus]",7,"[1.0/1.0, 2.0/1.0, 6.0/1.0, 7.0/1.0, 11.0/3.0, 14.0/1.0, 17.0/1.0]","[P-5, P-4, P-5, P8, P5, P-8]","[8.0, 32.0, 8.0, 36.0, 20.0, 24.0]",False,False,None,0,"[0.0, 8.0, 40.0, 48.0, 84.0, 104.0, 128.0]",0.000000
1,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F19%2C20%2C21%2C22%2C23%2C25%2C26%2C27%2C28%2C29%2C30%2C32%2C33%2C35%2C36%2F3%2C3%2C1%2C1%2B2%2C2%2C3%2C3%2C4%2C4%2C2%2C2%2C1%2C1%2C4%2C4%2F%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2B%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2C%402.0-end%2C%401-2.0%2C%402.0-end%2C%401-2.0%2Fhighlight&measure_range=19-36&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-3, 2, 2)]","[Tenor, Cantus, Altus, Tenor, Bassus, Altus, Cantus, Bassus]",8,"[19.0/4.0, 21.0/4.0, 22.0/4.0, 25.0/4.0, 27.0/4.0, 29.0/4.0, 32.0/2.0, 35.0/2.0]","[P8, P-5, P-4, P-5, P8, P8, P-11]","[16.0, 8.0, 24.0, 16.0, 16.0, 20.0, 24.0]",False,False,None,0,"[150.0, 166.0, 174.0, 198.0, 214.0, 230.0, 250.0, 274.0]",0.284091
2,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,FUGA,"[(1, 1, -3)]","[Altus, Tenor, Cantus, Bassus, Altus, Cantus, Bassus]",7,"[38.0/2.0, 38.0/4.0, 40.0/2.0, 42.0/2.0, 44.0/1.0, 46.0/2.0, 48.0/4.0]","[P-5, P8, P-12, P8, P8, P-11]","[4.0, 12.0, 16.0, 14.0, 18.0, 20.0]",False,False,None,0,"[298.0, 302.0, 314.0, 330.0, 344.0, 362.0, 382.0]",0.564394
3,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F50%2C51%2C52%2C53%2C54%2C55%2C56%2C57%2C59%2C60%2C61%2C62%2C63%2C64%2C65%2F2%2C1%2B2%2C1%2C4%2C4%2C3%2C2%2B3%2C2%2C4%2C4%2C1%2C1%2B2%2C2%2C3%2C3%2F%402.0-end%2C%404.0-end%2B%401-2.0%2C%401-4.0%2C%402.0-end%2C%401-2.0%2C%402.0-end%2C%404.0-end%2B%401-2.0%2C%401-4.0%2C%40all%2C%401-2.0%2C%404.0-end%2C%401-4.0%2B%402.0-end%2C%401-2.0%2C%402.0-end%2C%401-2.0%2Fhighlight&measure_range=50-65&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-2, 2, 1)]","[Altus, Cantus, Bassus, Tenor, Altus, Bassus, Cantus, Altus, Tenor]",9,"[50.0/2.0, 51.0/4.0, 53.0/2.0, 55.0/2.0, 56.0/4.0, 59.0/1.0, 61.0/4.0, 62.0/2.0, 64.0/2.0]","[P5, P-8, P4, P5, P-8, P8, P-4, P-5]","[12.0, 12.0, 16.0, 12.0, 18.0, 22.0, 4.0, 16.0]",False,False,None,0,"[394.0, 406.0, 418.0, 434.0, 446.0, 464.0, 486.0, 490.0, 506.0]",0.746212


# 6. Cadences

Here we use the cadence tool to report cadences in our piece. 

Note that there are many other metadata features about the cadences we can include!


## 6b.  Cadences with Linked Examples

Here we use the linkExample method to show the results.

Note that there are many other metadata features about the cadences we can include!


In [26]:
baseUrl = 'https://crimproject.org/mei/'

# now specify YOUR piece:
my_piece = 'CRIM_Model_0019.mei'


# and combine them
piece_url = baseUrl + my_piece

# load the piece from the web
piece = importScore(piece_url)

# find cadences
cadences = piece.cadences(keep_keys=False, voice_detail=True, key_sig=True, include_final=True)


cadences['Composer'] = piece.metadata['composer']
cadences['Title'] = piece.metadata['title']
cols = ['Composer', 'Title', 'CadType', 'Tone', 'CVFs', 'PartMap', 'Measure', 'Beat', 'Progress', 'Final', 'KeySig']
cadences = cadences[cols]
# and return the results with a link to the example
piece.linkExamples(df = cadences, piece_url=piece_url)

,Composer,Title,CadType,Tone,CVFs,PartMap,Measure,Beat,Progress,Final,KeySig
Last,,,,,,,,,,,
48.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Clausula Vera,G,TCu,"{'T': ['1'], 'C': ['2'], 'u': ['3']}",7.0,1.0,0.090909,G3,0.0
84.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,G,tC,"{'t': ['3'], 'C': ['4']}",11.0,3.0,0.159091,G3,0.0
136.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Quince,G,QC,"{'Q': ['1'], 'C': ['2']}",18.0,1.0,0.257576,G3,0.0
152.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Authentic,G,TCB,"{'T': ['1'], 'C': ['2'], 'B': ['4']}",20.0,1.0,0.287879,G3,0.0
224.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Abandoned Clausula Vera,C,zC,"{'z': ['2'], 'C': ['3']}",29.0,1.0,0.424242,G3,0.0
252.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,C,tC,"{'t': ['2'], 'C': ['4']}",32.0,3.0,0.477273,G3,0.0
286.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,NaN,Tc,"{'T': ['2'], 'c': ['3']}",36.0,4.0,0.541667,G3,0.0
296.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Authentic,C,CTB,"{'C': ['1'], 'T': ['3'], 'B': ['4']}",38.0,1.0,0.560606,G3,0.0
328.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,A,tC,"{'t': ['1'], 'C': ['2']}",42.0,1.0,0.621212,G3,0.0


## 6d.  Cadence Progress

The cadences in succession.

In [27]:
fig = px.scatter(
    cadences,
    x='Progress',
    y='Tone',
    color='CadType',
    color_discrete_map=cadence_color_dict,
)

fig.update_traces(marker=dict(size=15))

fig.update_layout(
    title=f"Progress of Cadences in {piece.metadata['composer']}: {piece.metadata['title']}",
    title_x=0.5,
    height=400,
    width=1000,
)

# Order the y-axis by the custom tone order
fig.update_yaxes(
    categoryorder='array',
    categoryarray=custom_tone_order,
)

fig.show()


## 6e.  Melodic Leaps:  Species

In [31]:
piece = importScore(path)
# cadences

limit_to_main_cadence_types = True
cadence_df = piece.cadences()
cadence_df['Composer'] = piece.metadata['composer'] 
cadence_df['Title'] = piece.metadata['title']
selected_types = ['Clausula Vera', 'Authentic', 'Phrygian Clausula Vera', 'Phrygian']
if limit_to_main_cadence_types:

    cadences_df = cadence_df[cadence_df['CadType'].isin(selected_types)]
else:
    pass
# 

# species
species_df = find_species(piece, limit_to_entries=False, species_intervals = ['5', '-5', '4', '-4'])

# plot
plot_species_cadences(species_df, cadence_df, 
                    piece.metadata['composer'], 
                    piece.metadata['title'])

# 7 Corpus Methods

## 7a Build a Corpus

Here we obtain the MEI files from our work.


```python
# now specify YOUR pieces:
my_pieces = ['https://crimproject.org/mei/CRIM_Model_0019.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_1.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_2.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_3.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_4.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_5.mei']


In [10]:

# now specify YOUR pieces:
my_pieces = ['https://crimproject.org/mei/CRIM_Model_0019.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_1.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_2.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_3.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_4.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_5.mei']

# file_list = sorted(glob.glob('MEI/LC*'))
corpus = CorpusBase(my_pieces)


In [11]:
# all the notes in the corpus

corpus.notes(combine_unisons_choice=True, combine_rests_choice=False, key_sig=True, include_final=True)


,KeySig,Composer,Title,Date,Final,1,2,3,4,5
0,0.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,None,G3,D5,Rest,Rest,Rest,NaN
1,0.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,None,G3,B4,G4,Rest,Rest,NaN
2,0.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,None,G3,D5,NaN,NaN,NaN,NaN
3,0.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,None,G3,E5,E4,Rest,Rest,NaN
4,0.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,None,G3,D5,G4,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2770,0.0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,None,G3,D5,NaN,NaN,D4,B2
2771,0.0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,None,G3,NaN,F4,NaN,NaN,NaN
2772,0.0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,None,G3,E5,E4,NaN,C4,C3
2773,0.0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,None,G3,C5,NaN,NaN,NaN,NaN


In [12]:
# all the notes, counted, and 'scaled' by counts

corpus.note_scaled(combine_unisons_choice=True, combine_rests_choice=False, key_sig=True, include_final=True)


,count,scaled_count,Composer,Title,Final,KeySig
A,57,0.04,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,G3,0
A3,34,0.03,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,G3,0
A4,2,0.00,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,G3,0
B,48,0.04,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,G3,0
B3,35,0.03,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,G3,0
...,...,...,...,...,...,...
F4,20,0.01,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,G3,0
G,61,0.04,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,G3,0
G#,1,0.00,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,G3,0
G3,44,0.03,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,G3,0


In [13]:
# weighted notes in the corpus

corpus_note_wt = corpus.note_weights(include_rests=True, key_sig=True, include_final=True)


In [14]:
# radar plot of notes in a corpus

radar_note_plot(corpus_note_wt,
                    pitch_class_order='Fifths',
                    exclude_rests=True,
                    limit_to_active=True,
                    color_grouping='title'
                    )

## 7b Mel, Har, and NGrams in Corpus

### Interval Settings

The settings:  here we specify the various kinds of intervals and length of ngrams we want to find:

```python
combine_unisons_choice = False
combine_rests_choice = False
kind_choice = 'q'
directed_choice = False
compound_choice = False
exclude_rests = True
ngram_length = 4
```

These are in turn used with the various corpus functions we defined at the start of our notebook (see above).  For example:

`corpus_melodic = corpus_mel(corpus, kind_choice, compound_choice, directed_choice)`

or

`corpus_harmonic = corpus_har(corpus, kind_choice, compound_choice, directed_choice)`

or

`corpus_ngrams = corpus_ngrams(corpus, ngram_length, combine_rests_choice)`



In [15]:
# settings:  here we specify the various kinds of intervals and length of ngrams we want to find
combine_unisons_choice = False
combine_rests_choice = False
kind_choice = 'd'
directed_choice = False
compound_choice = False
exclude_rests = True
ngram_length = 4

## 7c. Corpus Notes as Barchart

Here we consider the *counts* of all the, voice by voice

In [16]:

# run function and create chart
nr = corpus.notes(corpus, combine_unisons_choice, combine_rests_choice)

id_vars = ['Composer', 'Title', 'Date']

# Use list comprehension to determine value_vars
value_vars = [col for col in nr.columns if col not in id_vars]

# Melt the DataFrame
melted_df = pd.melt(
    nr,
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Voice',
    value_name='Note'
)

melted_df['Note'] = melted_df['Note'].replace('NaN', np.nan)

# Remove NaN values for the visualization
melted_df_clean = melted_df.dropna(subset=['Note'])

# Create a basic bar chart showing the count of each note
chart_title = "Distribution of Notes in Corpus"

chosen_feature_to_compare = 'Title'

# Pre-aggregate the data (similar to your working example)
grouped_data = melted_df_clean.groupby(['Note', chosen_feature_to_compare]).size().reset_index(name='count')

# Create the bar chart with the pre-aggregated data
fig = px.bar(
    grouped_data,
    x='Note',
    y='count',
    color=chosen_feature_to_compare,
    barmode='stack',  # or 'group' if you prefer stacked bars
    color_discrete_sequence=px.colors.qualitative.Bold,  # Bold color scheme
    opacity=1.0,  # Full opacity for vibrant colors
    title=chart_title,
    category_orders={'Note': pitch_order}
)

# Update layout for clean appearance
fig.update_layout(
    xaxis_title="Note", 
    yaxis_title="Count",
    legend_title='Voice',
    height=500,
    width=900,
    plot_bgcolor='white',  # White background
    paper_bgcolor='white',  # White paper background
    xaxis={'categoryorder': 'array', 'categoryarray': pitch_order}
)

# Minimize grid lines
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5)

fig.show()

## 7d.  Voice Range Charts

Here we use weighted notes to get the "top N" notes in each voice



In [19]:
# get the notes and durs for tenors, with finals
report_by_final = True
report_by_voice = False

corpus_notes_and_durs = corpus.note_durs(pitch_class=False, aggregate=False, key_sig=False, include_final=True)


# standarize pitch names for flats and sharps
corpus_notes_and_durs['Pitch_Class']= corpus_notes_and_durs['Notes'].map(standardize_note)
corpus_notes_and_durs['Composer']= corpus_notes_and_durs['Composer'].fillna('Anon')

len(corpus_notes_and_durs.Title.unique())

# we already created corpus_notes_and_durs above!

# get sum of durations for each voice in each piece--these should be the same!
grouped_notes_durs =  corpus_notes_and_durs.groupby(['Composer', 'Title', 'Voice'])['Durs'].sum().reset_index()
grouped_notes_durs.columns = ['Composer', 'Title', 'Voice', 'TotalDuration']
    
# Merge the original dataframe with the grouped results
corpus_pitch_data = pd.merge(corpus_notes_and_durs, grouped_notes_durs[['Composer', 'Title', 'Voice', 'TotalDuration']], 
                         on=['Composer', 'Title', 'Voice'], how='left')

corpus_pitch_data['Dur_Ratio'] = (corpus_pitch_data['Durs']/corpus_pitch_data['TotalDuration']).round(3)
corpus_pitch_data['Percentage'] = corpus_pitch_data['Dur_Ratio'] * 100
corpus_pitch_data = corpus_pitch_data.groupby(['Composer', 'Title', 'Voice', 'Notes', 'Final'])['Percentage'].sum().reset_index()
corpus_pitch_data

,Composer,Title,Voice,Notes,Final,Percentage
0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,1,A4,G,11.6
1,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,1,B4,G,7.1
2,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,1,C5,G,15.7
3,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,1,D5,G,19.3
4,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,1,E5,G,10.1
...,...,...,...,...,...,...
317,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,4,E4,G,0.7
318,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,4,F#3,G,0.4
319,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,4,F3,G,7.8
320,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,4,G3,G,15.8


In [20]:
voice_range_chart(corpus_pitch_data, recta_order)

## 7e.  Corpus Ngrams Maps

Here we produce maps of the ngrams in each piece, in pairs:  model + mass movement.



In [138]:
# the ngram conditions
entries_only = True
combineUnisons = False
kind_choice ='d'
directed = True
compound = False
thematic = True
anywhere = True
n = 3
include_count = True


model = importScore(my_pieces[0])
nr = model.notes(combineUnisons=combineUnisons)
mel = model.melodic(df=nr, kind=kind_choice, compound=False, unit=0, end=False)
mod_mel_ngrams = model.ngrams(df=mel, n=n, exclude=['Rest'])
mod_entry_ngrams = model.entries(df=mel, n=n, thematic=thematic, anywhere=anywhere, exclude=['Rest'])
mod_mel_ngrams_duration = model.durations(df=mel, n=n, mask_df=mod_entry_ngrams)
mod_entries_stack = list(mod_entry_ngrams.stack().unique())

# find entries mass movements.  
# here we will display the results so that the motet is compared with each Kyrie, then each Gloria, etc
# in each case Lasso's Mass Movement will come first, then Daser's

mass_movements = my_pieces[1:]
for movement in mass_movements:       
    m_movement = importScore(movement)
    nr = m_movement.notes(combineUnisons=combineUnisons)
    mel = m_movement.melodic(df=nr, kind=kind_choice, compound=False, unit=0, end=False)
    mass_mvmt_mel_ngrams = m_movement.ngrams(df=mel, n=n)
    mass_mvmt_entries = m_movement.entries(df=mel, n=n, thematic=thematic, anywhere=anywhere)
    mass_mvmt_mel_ngrams_duration = m_movement.durations(df=mel, n=n, mask_df=mass_mvmt_entries)
    mass_mvmt_entries_stack = mass_mvmt_entries.stack()
      
    # find shared entries
    shared_entries = list(set(mass_mvmt_entries_stack).intersection(mod_entries_stack))
    
    # print model metadata and chart
    print(model.metadata['composer'])
    print(model.metadata['title'])
    
    display(viz.plot_ngrams_heatmap(mod_entry_ngrams, 
                                    mod_mel_ngrams_duration, 
                        selected_patterns=shared_entries,
                        voices=[], 
                                    includeCount=False))
      
    # # print mass movement metadata and chart
    print(m_movement.metadata['composer'])
    print(m_movement.metadata['title'])
    display(viz.plot_ngrams_heatmap(mass_mvmt_entries, 
                                    mass_mvmt_mel_ngrams_duration, 
                                selected_patterns=shared_entries,
                                voices=[], 
                                    includeCount=False)) 
        

Giovanni Pierluigi da Palestrina
Veni sponsa Christi


alt.Chart(...)

Giovanni Pierluigi da Palestrina
Missa Veni sponsa Christi: Kyrie


alt.Chart(...)

Giovanni Pierluigi da Palestrina
Veni sponsa Christi


alt.Chart(...)

... (10 outputs left out)

Giovanni Pierluigi da Palestrina
Veni sponsa Christi


alt.Chart(...)

Giovanni Pierluigi da Palestrina
Missa Veni sponsa Christi: Agnus Dei


alt.Chart(...)

## 7f.  Corpus:  Sonority Ngrams

Buiding Sonority Ngrams

In [29]:
# now specify YOUR pieces:
my_pieces = ['https://crimproject.org/mei/CRIM_Model_0012.mei']
my_pieces

corpus = CorpusBase(my_pieces)


In [36]:
ngram_length = 1
sort= True
minimum_beat_strength = 0.0

sonorities_low_lines = corpus.sonority_ngrams(ngram_length=ngram_length, 
                                    metadata_choice=True, 
                                    include_offset=False, 
                                    include_progress=True, 
                                    compound=True, 
                                    sort=sort, 
                                    minimum_beat_strength=minimum_beat_strength, 
                                    key_sig=False, 
                                    include_final=False)

def convert_tuple(tup):
    """
    Convert tuple to string.

    Parameters:
    -----------
    tup : tuple
        Tuple to convert

    Returns:
    --------
    str
        Converted string
    """
    out = ""
    if isinstance(tup, tuple):
        out = '_'.join(tup)
    return out

sonorities_low_lines['Sonority'] = sonorities_low_lines['Sonority'].map(convert_tuple)
sonorities_low_lines['Low Line'] = sonorities_low_lines['Low Line'].map(convert_tuple)
sonorities_low_lines

,Measure,Beat,Progress,Composer,Title,Date,Sonority,Low Line,Low_Sonority
0,1.0,1.0,0.000000,Pierre Sandrin,Voulant honneur,None,10/5,-M2,-M2_10/5
1,1.0,3.0,0.010101,Pierre Sandrin,Voulant honneur,None,12/8/3,-P5,-P5_12/8/3
2,1.0,4.0,0.015152,Pierre Sandrin,Voulant honneur,None,12/8/4,Held,Held_12/8/4
3,2.0,1.0,0.020202,Pierre Sandrin,Voulant honneur,None,12/8/4,Held,Held_12/8/4
4,2.0,1.5,0.022727,Pierre Sandrin,Voulant honneur,None,12/8/3,Held,Held_12/8/3
...,...,...,...,...,...,...,...,...,...
259,49.0,4.0,0.984848,Pierre Sandrin,Voulant honneur,None,10/6/4,-M2,-M2_10/6/4
260,49.0,4.5,0.987374,Pierre Sandrin,Voulant honneur,None,10/6/3,Held,Held_10/6/3
261,50.0,1.0,0.989899,Pierre Sandrin,Voulant honneur,None,11/8/5,-P5,-P5_11/8/5
262,50.0,2.0,0.994949,Pierre Sandrin,Voulant honneur,None,10/8/5,Held,Held_10/8/5


In [ ]:
# Customize the size and title
size = 10
title_font = {'size': 12}

# Create the scatter plot

fig = px.scatter(sonorities_low_lines,
                 x='Progress',
                 y='Low_Sonority',
                 color='Title',
                 hover_data=['Composer', 'Title', 'Measure', 'Beat', 'Progress', 'Low_Sonority'],
                 title='Scatter Plot of Sonority vs Progress')

# Customize the layout
fig.update_layout(
    title_font=title_font,
    xaxis_title='Progress',
    yaxis_title='Low Sonority',
    legend_title='Title',
	height=600,
	width=1000,
)

# Show the plot
fig.show()

# Save the plot as an HTML file
output_file = "sonority_progress_scatter.html"
fig.write_html(output_file)